In [23]:
import sys
sys.path.append('../')
from source import Node 
from source import Align
import glob
from pathlib import Path
import networkx as nx
from ipysigma import Sigma
import pandas as pd

In [24]:
# Node.PrepareGenomeQueryFasta(
#     genomes_dir="../data/genomes"
# )

In [25]:
FastaFiles = glob.glob("../data/processed/selected_components/*.fasta")
FastaFiles

['../data/processed/selected_components/ProblematicComponents.TargetClass.aminocoumarin.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.aminoglycoside.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.bacitracin.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.beta-lactam.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.colistin.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.drug_and_biocide_and_metal_resistance.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.drug_and_biocide_resistance.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.elfamycin.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.fluoroquinolone.fasta',
 '../data/processed/selected_components/ProblematicComponents.TargetClass.fosfomycin.fasta',
 '../data/proces

In [26]:
for file in FastaFiles:
    name = file.split("/")[-1].split(".")[-2]
    if not Path(f"../data/processed/diamond_db/{name}.dmnd").is_file():
        Node.MakeDiamondDB(
        fasta_dir=file,
        db_dir="../data/processed/diamond_db/",
        db_name = name)
    print("All databases are already created")

All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
Running: diamond makedb --in ../data/processed/selected_components/ProblematicComponents.TargetClass.GenomeProtein.fasta --db ../data/processed/diamond_db/GenomeProtein --quiet
DIAMOND database saved as: GenomeProtein.dmnd
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created
All databases are already created


In [27]:
DiamondDatabase = glob.glob("../data/processed/diamond_db/*.dmnd")
DiamondDatabase

['../data/processed/diamond_db/aminocoumarin.dmnd',
 '../data/processed/diamond_db/aminoglycoside.dmnd',
 '../data/processed/diamond_db/bacitracin.dmnd',
 '../data/processed/diamond_db/beta-lactam.dmnd',
 '../data/processed/diamond_db/colistin.dmnd',
 '../data/processed/diamond_db/drug_and_biocide_and_metal_resistance.dmnd',
 '../data/processed/diamond_db/drug_and_biocide_resistance.dmnd',
 '../data/processed/diamond_db/elfamycin.dmnd',
 '../data/processed/diamond_db/fluoroquinolone.dmnd',
 '../data/processed/diamond_db/fosfomycin.dmnd',
 '../data/processed/diamond_db/fosmidomycin.dmnd',
 '../data/processed/diamond_db/GenomeProtein.dmnd',
 '../data/processed/diamond_db/glycopeptide.dmnd',
 '../data/processed/diamond_db/macrolide.dmnd',
 '../data/processed/diamond_db/multidrug.dmnd',
 '../data/processed/diamond_db/phosphonic_acid.dmnd',
 '../data/processed/diamond_db/polymyxin.dmnd',
 '../data/processed/diamond_db/quinolone.dmnd',
 '../data/processed/diamond_db/rifampin.dmnd',
 '../data

In [28]:
for database in DiamondDatabase:
    name = database.split("/")[-1].split(".")[0]
    name = name.replace(" ","_")
    if not Path(f"../data/processed/diamond_results/NodeProteinAlignment.{name}.tsv").is_file():
        Align.RunDiamond(
            query="../data/genomes/all_genomes_query.fasta",
            db=database,
            output=f"../data/processed/diamond_results/NodeProteinAlignment.{name}.tsv",
            maxseq=6,
            qcov = 80,
            minid = 90
            )
    else:
        print("All alignments done!")

All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!
All alignments done!


In [22]:
color_map = {
    'multidrug':      "#FD6F63",     
    'beta-lactam':    "#89BDE7",  
    'bacitracin':     "#2E8D19",    
    'tetracycline':   "#CD73E8",
    'glycopeptide':   "#F3BC71",  
    'macrolide':      "#D4D451",     
    'aminoglycoside': "#ECAD2F", 
    'quinolone':      "#14E974",
    'drug_and_biocide_resistance': "#EA17B9",
    "drug_and_biocide_and_metal_resistance": "#EA17B9",
    'fosfomycin':     "#6063F4",
    'fluoroquinolone':"#60D4F4",
    'elfamycin':      "#7B079E",
    'aminocoumarin':  "#EE00F6",
    'rifampin':       "#0BF8E8",
    'phosphonic acid':"#262897",
    'fosmidomycin':   "#7B5404",
    'colistin' :      "#7FBE03",
    'polymyxin':      "#97B262",
    'GenomeProtein':"#534F51",
    }

In [29]:
TargetClass = "multidrug"
ProblematicComponentsByClass = nx.read_graphml(f"../data/processed/ProblematicComponentsByClass.{TargetClass}.graphml")

In [30]:
GenomeNodesAligned = pd.read_csv(
    f"../data/processed/diamond_results/NodeProteinAlignment.{TargetClass}.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid stitle pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)
GenomeNodesAligned["sseqid"] = GenomeNodesAligned.sseqid.apply(lambda x: x.split("|")[0])

In [31]:
ProblematicComponentsByClass_AndGenomes = nx.Graph(ProblematicComponentsByClass)
GenomeNodesDict = {}
NewNodes = []
for _,i in GenomeNodesAligned.iterrows():
    ProblematicComponentsByClass_AndGenomes.add_edge(i["sseqid"],i["qseqid"], pident = i["pident"])
    GenomeNodesDict[i["qseqid"]] = {"Drug Class":"GenomeProtein", "Sequence":i["full_qseq"], "Name":i["qseqid"].split("|")[-1]}
nx.set_node_attributes(ProblematicComponentsByClass_AndGenomes, GenomeNodesDict)

In [32]:
meu_layout = {
    "scalingRatio": 50.0,           # Aumente para afastar os grupos
    "gravity": 0.4,                 # Reduza para não amontoar no centro
    "repulsion": 2,               # Aumente para afastar os nós
    # "outboundAttractionDistribution": True, # Empurra hubs para fora
    # "barnesHutOptimize": True,      # Essencial para seus 70k nós
    # "linLogMode": True              # Melhora a definição de clusters biológicos 
}
Sigma(
    ProblematicComponentsByClass_AndGenomes, 
    node_size  = ProblematicComponentsByClass_AndGenomes.degree(),
    node_color =  [ProblematicComponentsByClass_AndGenomes.nodes[n].get("Drug Class") for n in ProblematicComponentsByClass_AndGenomes.nodes()],
    raw_node_color=True,
    node_color_palette = color_map,
    default_edge_type = "curve",
    layout_settings=meu_layout,
    start_layout=30,
    )

Sigma(nx.Graph with 6,196 nodes and 18,964 edges)

In [33]:
NodeBait = "CARD_1668"

StringFasta = []
for component in list(nx.connected_components(ProblematicComponentsByClass_AndGenomes)):
    if NodeBait.strip() in component:
        for nodes in component:
            DrugClass = ProblematicComponentsByClass_AndGenomes.nodes[nodes]['Drug Class']
            Sequence = ProblematicComponentsByClass_AndGenomes.nodes[nodes]['Sequence']
            Name = ProblematicComponentsByClass_AndGenomes.nodes[nodes]['Name']
            StringFasta.append(f">{nodes}|{DrugClass}|{Name}\n{Sequence}")
AlignedComponent = Align.ProteinAligner(StringFasta)
for AlignedSeq in AlignedComponent:
    print(f">{AlignedSeq.id.strip()}|len:{len(AlignedSeq.seq.replace('-',''))}")
    print(AlignedSeq.seq)

ValueError: No records found in handle